# Исследовательский анализ рынка общественного питания Москвы


### Цели и задачи проекта

*Цель проекта* - провести исследовательский анализ рынка Москвы для помощи инвесторам в выборе подходящего места для открытия заведения общественного питания (кафе, ресторана или бара) и формирования меню и цен. 

### Описание данных

Файл /datasets/rest_info.csv содержит информацию о заведениях общественного питания:
- name — название заведения;
- address — адрес заведения;
- district — административный район, в котором находится заведение, например Центральный административный округ;
- category — категория заведения, например «кафе», «пиццерия» или «кофейня»;
- hours — информация о днях и часах работы;
- rating — рейтинг заведения по оценкам пользователей в Яндекс Картах (высшая оценка — 5.0);
- chain — число, выраженное 0 или 1, которое показывает, является ли заведение сетевым (для маленьких сетей могут встречаться ошибки):
  - 0 — заведение не является сетевым;
  - 1 — заведение является сетевым.
- seats — количество посадочных мест.

Файл /datasets/rest_price.csv содержит информацию о среднем чеке в заведениях общественного питания:
- price — категория цен в заведении, например «средние», «ниже среднего», «выше среднего» и так далее;
- avg_bill — хранит среднюю стоимость заказа в виде диапазона, например:
  - «Средний счёт: 1000–1500 ₽»;
  - «Цена чашки капучино: 130–220 ₽»;
  - «Цена бокала пива: 400–600 ₽».
    и так далее;
- middle_avg_bill — число с оценкой среднего чека, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Средний счёт»:
  - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
  - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
  - Если значения нет или оно не начинается с подстроки «Средний счёт», то в столбец ничего не войдёт.
- middle_coffee_cup — число с оценкой одной чашки капучино, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Цена одной чашки капучино»:
  - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
  - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
  - Если значения нет или оно не начинается с подстроки «Цена одной чашки капучино», то в столбец ничего не войдёт.

### Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Предобработка данных
3. Исследовательский анализ данных
4. Итоговый вывод и рекомендации
---

## 1. Загрузка данных и знакомство с ними


In [1]:
# Импортируем библиотеку
import pandas as pd

# Загружаем библиотеки для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

!pip install phik

# Загружаем библиотеку для расчёта коэффициента корреляции phi_k
from phik import phik_matrix

ModuleNotFoundError: No module named 'scipy.stats._mvn'

In [ ]:
# Выгружаем данные в переменные bank_df и clients_df
info_df = pd.read_csv('https://code.s3.yandex.net/datasets/rest_info.csv')
price_df = pd.read_csv('https://code.s3.yandex.net/datasets/rest_price.csv')

In [ ]:
info_df.info()

In [ ]:
info_df.head()

In [ ]:
price_df.info()

In [ ]:
price_df.head()

---

### Промежуточный вывод


**rest_info.csv:**
- В датасете *rest_info.csv* 9 столбцов и 8406 строк.
- Значения в столбцах соответствуют своему описанию.
- Все названия столбцов приведены к единому стилю.
- Типы данных представлены object, float64 и int64.
- Можно оптимизировать целочисленный тип данных столбца chain.
- Пропуски присутствуют в двух столбцах (hours и seats).

**rest_price.csv:**
- В датасете *rest_price.csv* 5 столбцов и 4058 строк.
- Значения в столбцах соответствуют своему описанию.
- Все названия столбцов приведены к единому стилю.
- Типы данных представлены object и float64.
- Для столбца price тип данных можно перевести в category.
- Пропуски присутствуют в четырех столбцах (price, avg_bill, middle_avg_bill и middle_coffee_cup).

### Подготовка единого датафрейма


In [ ]:
#Объединяем датасеты
df = pd.merge(info_df, price_df, on='id', how='left')

In [ ]:
df.info()

## 2. Предобработка данных



In [ ]:
#Оптимизируем целочисленный тип данных столбца seats
df['seats'] = pd.to_numeric(df['seats'], errors='coerce', downcast='integer')

In [ ]:
#Оптимизируем целочисленный тип данных столбца chain
df['chain'] = pd.to_numeric(df['chain'], errors='coerce', downcast='integer')

In [ ]:
#Меняем тип данных столбца price на category
df['price'] = df['price'].astype('category')

In [ ]:
#Выведем новый тип данных
df.info()

In [ ]:
#Кол-во пропущенных строк в датафрейме
df.isna().sum()

In [ ]:
#Процент пропущенных строк
df.isna().sum() / len(df) * 100

Данные пропущены в 6 столбцах ('hours', 'seats', 'price', 'avg_bill', 'middle_avg_bill', 'middle_coffee_cup')
- Максимальное значение пропущенных данных в столбце 'middle_coffee_cup' - 93,6%, на втором месте столбец 'middle_avg_bill' - 62,5%, на третем месте столбец 'price' - 60,5%
- Пропуски в вышеуказанных столбцах не критичны для нашего исследовательского анализа, поэтому их можно игнорировать.

In [ ]:
#Проверка явных дубликатов до нормализации данных
df.duplicated().sum()

In [ ]:
df.duplicated(subset=['name', 'address']).sum()

In [ ]:
#Нормализация данных в столбце 'name' с названием заведения
#df['name'] = df['name'].str.lower().str.replace(' ','_')
df['name'] = df['name'].str.lower().str.strip()
df['address'] = df['address'].str.lower().str.strip()

In [ ]:
#Явные дубликаты в столбце 'name' и 'address'
df.duplicated(subset=['name', 'address'], keep='first')

In [ ]:
#Выведем явные дубликаты
df[df[['name', 'address']].duplicated() == True]

In [ ]:
#Кол-во дубликатов
df.duplicated(subset=['name', 'address'], keep='first').sum()

In [ ]:
#Удаляем явные дубликаты
df = df.drop_duplicates(subset=['name', 'address'], keep='first')

In [ ]:
df.info()

- Для дальнейшей работы создаем столбец `is_24_7` с обозначением того, что заведение работает ежедневно и круглосуточно, то есть 24/7:
  - логическое значение `True` — если заведение работает ежедневно и круглосуточно;
  - логическое значение `False` — в противоположном случае.

In [ ]:
df['is_24_7'] = False
df.loc[(df['hours'].str.contains('ежедневно',na=False) & df['hours'].str.contains('круглосуточно',na=False)),'is_24_7'] = True

In [ ]:
df['is_24_7'].value_counts()

In [ ]:
df.info()

---

### Промежуточный вывод


Были загружены и объеденены данные из 2х датасетов /datasets/rest_info.csv и /datasets/rest_price.csv. 
Объендененный датасет содержит 13 столбцов и 8406 строк, в которых представлена информация о заведениях общественного питания и о среднем чеке в данных заведениях.

- В шести стобцах (hours, seats, price, avg_bill, middle_avg_bill и middle_coffee_cup) были обнаружены пропущенные значения.
- Максимальное значение пропущенных данных в столбце 'middle_coffee_cup' - 93,6%, на втором месте столбец 'middle_avg_bill' - 62,5%, на третем месте столбец 'price' - 60,5%.
- Для оптимизации работы с данными в датафрейме были произведены следующие изменения типов данных:
  - 'chain': тип данных изменён с int64 на int8.
  - 'price': тип данных изменён с object на category.
- Проведена нормализация данных с текстовыми значениями с целью исключения неявных дубликатов, а также выявлено 4 явных дубликата.
- После удаления явных дубликатов осталось 8402 строк.
- Создан столбец is_24_7 с обозначением того, что заведение работает ежедневно и круглосуточно.

## 3. Исследовательский анализ данных


---

### Задача 1

Какие категории заведений представлены в данных? Исследуйте количество объектов общественного питания по каждой категории. Результат сопроводите подходящей визуализацией.

In [ ]:
#Количество заведений общественного питания в абсолютных значениях
df['category'].value_counts()

In [ ]:
#Количества заведений общественного питания в относительных значениях
df['category'].value_counts(normalize=True)

In [ ]:
df['category'].value_counts().plot(kind='bar',
                title='Количество объектов общественного питания по каждой категории',
                legend=False,
                ylabel='Количество объектов',
                xlabel='Категории',
                rot=90)

plt.show()

- Больше всего заведений представлены в категории кафе, на втором месте - ресторан, на третем месте - кофейня
- Доля кафе среди всех категорий составляет 28%, доля ресторанов - 24%, доля кофеен - 17%
- Меньше всего заведений в категориях столовая и булочная, доля которых составляет 4% и 3% соответственно. 

---

### Задача 2

Какие административные районы Москвы присутствуют в данных? Исследуйте распределение количества заведений по административным районам Москвы, а также отдельно распределение заведений каждой категории в Центральном административном округе Москвы. Результат сопроводите подходящими визуализациями.

In [ ]:
#Количество заведений по административным районам в абсолютных значениях
df['district'].value_counts()

In [ ]:
#Количество заведений по административным районам в относительных значениях
df['district'].value_counts(normalize=True)

In [ ]:
#Районы Москвы
df['district'].value_counts(normalize=True).plot(kind='barh',
                title='Административные районы Москвы',
                legend=False,
                ylabel='Количество заведений',
                xlabel='Районы',
                rot=0)

plt.show()

In [ ]:
#Количество заведений по категориям в ЦАО в абсолютных значениях
df.groupby('district')['category'].value_counts()

In [ ]:
#Количество заведений по категориям в ЦАО в относительных значениях
df.groupby('district')['category'].value_counts(normalize=True)

In [ ]:
#Распределение количества заведений по административным районам Москвы
df_distr = df.groupby('district')['category'].value_counts().unstack(fill_value=0)

df_distr.plot(kind='bar',
                title='Распределение количества заведений по административным районам Москвы',
                legend=True,
                ylabel='Количество заведений',
                xlabel='Категории',
                rot=90, 
                figsize=(12, 8))

plt.show()

In [ ]:
#Распределение заведений каждой категории в Центральном административном округе Москвы
df_distr = df.groupby('district')['category'].value_counts().unstack(fill_value=0)
centr_df = df_distr.loc['Центральный административный округ'].sort_values(ascending=False)

centr_df.plot(kind='bar',
                title='Распределение количества заведений по Центральному административному округу Москвы',
                legend=True,
                ylabel='Количество заведений',
                xlabel='Категории',
                rot=90, 
                figsize=(12, 8))

plt.show()

- Большая доля заведений находится в Центральном районе Москвы и составляет 27%.
- Самое маленькое кол-во заведений находится в Северно-Западном районе Москвы (доля заведений 5%) 
- В центральном районе топ-3 категории заведений: ресторан, кафе, кофейня. 
- Самые низкие позиции по кол-ву заведений в центральном районе занимают: булочная, столовая, быстрое питание и пиццерия. 

---

### Задача 3

Изучите соотношение сетевых и несетевых заведений в целом по всем данным и в разрезе категорий заведения. Каких заведений больше — сетевых или несетевых? Какие категории заведений чаще являются сетевыми? Исследуйте данные, ответьте на вопросы и постройте необходимые визуализации.

In [ ]:
#Соотношение сетевых и несетевых заведений в целом по всем данным
df['chain'].value_counts(normalize=True).plot(kind='bar',
                title='Соотношение сетевых и несетевых заведений в целом по всем данным',
                legend=False,
                ylabel='Количество заведений',
                xlabel='Тип заведения: 1 - сетевое, 0 - не сетевое',
                rot=0)
plt.show()

In [ ]:
#Соотношение сетевых и несетевых заведений в разрезе категорий заведения
df_unstack = df.groupby('category')['chain'].value_counts(normalize=True).unstack(fill_value=0)

df_unstack.plot(kind='bar',
                title='Соотношение сетевых и несетевых заведений в разрезе категорий заведения',
                legend=True,
                ylabel='Количество заведений',
                xlabel='Категории',
                rot=90)

plt.show()

In [ ]:
df_share = df.groupby('category')['chain'].mean().sort_values(ascending=False)
print(df_share)

In [ ]:
#Доля сетевых заведений по категориям
df_share = df.groupby('category')['chain'].mean().sort_values(ascending=False)

df_share.plot(
    kind='bar',
    title='Доля сетевых заведений по категориям',
    ylabel='Доля сетевых заведений',
    xlabel='Категории',
    rot=45
)

plt.show()

- В разрезе всех данных по количеству больше несетевых заведений, чем сетевых
- Категории заведений, которые чаще являются сетевыми: булочная (доля - 61%), пиццерия (доля - 52%), кофейня (доля - 51%)

---

### Задача 4

Исследуйте количество посадочных мест в заведениях. Встречаются ли в данных аномальные значения или выбросы? Если да, то с чем они могут быть связаны? Приведите для каждой категории заведений наиболее типичное для него количество посадочных мест. Результат сопроводите подходящими визуализациями.


In [ ]:
#Количество посадочных мест в заведениях
category_seats = df.groupby('category')['seats'].sum().plot(kind='bar',
                title='Количество посадочных мест в заведениях',
                legend=False,
                ylabel='Количество посадочных мест',
                xlabel='Заведения',
                rot=90)

In [ ]:
#Статистические показатели столбца seats
df['seats'].describe()

- Среднее значение 108.36 и медиана 75.0 находятся достаточно далеко друг от друга, что указывает на несимметричное распределение данных и наличие выраженных выбросов
- Минимальное кол-во метс - 0. Данные выбросы могут быть связаны с отсутствием внесенных данных, а также с небольшими несетевыми заведениями на небольшое кол-во мест. 
- Максимальное кол-во мест - 1288, что может быть связано с единичными большими заведениями, фуд-кортами, заведениями питания в спортивно-концертных комплексах, либо с неправильно заполненной информацией о кол-ве мест, что также является выбросами. 
- Значения меньшя 25го процентиля (40) и выше 75го процентиля (140) - можно считать выбросами. 

In [ ]:
#Проиллюстрируем выбросы
df.boxplot(column='seats', vert=False)
plt.title('Распределение значений количества мест')
plt.xlabel('Количество мест')

plt.show() 

- Распределение кол-ва мест асимметричное, скошенное вправо. 

In [ ]:
#Типичное количество посадочных мест для каждой категории заведений
median_category = df.groupby('category')['seats'].median().sort_values(ascending=False)

print(median_category)

In [ ]:
median_category = df.groupby('category')['seats'].median().sort_values(ascending=False).plot(kind='bar',
                title='Количество посадочных мест для каждой категории заведений',
                legend=False,
                ylabel='Количество посадочных мест',
                xlabel='Категории заведений',
                rot=90)

- Больше всего посадочных мест в категории рестораны, бар/паб, кофейня, меньше всего в категории булочная. 

---

### Задача 5

Исследуйте рейтинг заведений. Визуализируйте распределение средних рейтингов по категориям заведений. Сильно ли различаются усреднённые рейтинги для разных типов общепита?

In [ ]:
print(df['rating'].describe())

In [ ]:
print(df.groupby('category')['rating'].describe())

In [ ]:
category_rating = df.groupby('category')['rating'].mean().sort_values(ascending=False)
print(category_rating)

In [ ]:
#Распределение средних рейтингов по категориям заведений
category_rating = df.groupby('category')['rating'].mean().sort_values(ascending=False).plot(kind='bar',
                title='Распределение средних рейтингов по категориям заведений',
                legend=False,
                ylabel='Рейтинг',
                xlabel='Заведения',
                rot=90)

- Средний рейтинг по Москве практически одинаков для всех категорий заведений и составлет 4+.
- Наивысший рейтинг у заведений в категории бар/паб (4.39), ниже всего рейтинг в категории быстрое питание (4.05)

---

### Задача 6

Изучите, с какими данными показывают самую сильную корреляцию рейтинги заведений? Постройте и визуализируйте матрицу корреляции рейтинга заведения с разными данными: его категория, положение (административный район Москвы), статус сетевого заведения, количество мест, ценовая категория и признак, является ли заведения круглосуточным. Выберите самую сильную связь и проверьте её.

In [ ]:
correlation_matrix = df[['rating', 'category', 'district', 'chain', 'seats', 'price', 'is_24_7']].phik_matrix()

correlation_matrix.loc[correlation_matrix.index != 'rating'][['rating']].sort_values(by='rating', ascending=False)

In [ ]:
plt.figure(figsize=(2, 6))

correlation_matrix = df[['rating', 'category', 'district', 'chain', 'seats', 'price', 'is_24_7']].phik_matrix()

data_heatmap = correlation_matrix.loc[correlation_matrix.index != 'rating'][['rating']].sort_values(by='rating', ascending=False)
sns.heatmap(data_heatmap,
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            linewidths=0.5, 
            cbar=False )

plt.title('Тепловая карта коэффициента phi_k \n для данных rating')
plt.xlabel('Рейтинг')

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from phik import report
features = ['category', 'district', 'chain', 'seats', 'price', 'is_24_7', 'rating']

correlation_matrix = df[features].phik_matrix()

plt.figure(figsize=(10, 8))
im = plt.imshow(correlation_matrix.values, cmap='coolwarm', vmin=0, vmax=1)

plt.xticks(np.arange(len(correlation_matrix.columns)), correlation_matrix.columns, rotation=45, ha='right')
plt.yticks(np.arange(len(correlation_matrix.columns)), correlation_matrix.columns)

for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[i, j]
        plt.text(j, i, f"{value:.2f}", ha='center', va='center', color='black', fontsize=8)

plt.title("Матрица корреляции признаков с рейтингом")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

corr_with_rating = correlation_matrix['rating'].drop('rating')
strongest_feature = corr_with_rating.abs().idxmax()
strength_value = correlation_matrix.loc[strongest_feature, 'rating']

print(f"Коэффициент phi_k: {strength_value:.2f}")

- Наибольшая корреляция рейтинга наблюдается с категорией цен в заведениях (0.22), административным районом (0.20) и категорией заведений (0.19). Для остальных признаков коэффициент корреляции снижается с 0.15 до 0.
- Между категорией заведения и категорией цены самая сильная связь (0.58), что означает, что тип заведения сильно влияет на его ценовую категорию. 

In [ ]:
#Расчет значения среднего рейтинга для разных ценовых категорий заведений
mean_rating = df.groupby('price')['rating'].mean()

price_order = ['низкие', 'средние', 'выше среднего', 'высокие']
mean_rating = mean_rating.reindex(price_order)

print(mean_rating)

In [ ]:
#Визуализация
plt.plot(mean_rating.index, mean_rating.values, marker='o')

plt.show()

- Средний рейтинг заведений увеличивается по мере роста ценовой категории.
- Самые высокие оценки получают заведения высокого ценового сегмента. Это может свидетельствовать о более высоком качестве сервиса, кухни и атмосферы в дорогих ресторанах.

---

### Задача 7

Сгруппируйте данные по названиям заведений и найдите топ-15 популярных сетей в Москве. Для них посчитайте значения среднего рейтинга. Под популярностью понимается количество заведений этой сети в регионе. К какой категории заведений они относятся? Результат сопроводите подходящими визуализациями.

In [ ]:
#Топ-15 популярных сетей в Москве
top_15 = (df[df['chain'] == 1].groupby('name').agg(name_count=('name', 'count'), avg_rating=('rating', 'mean')).sort_values(by='name_count', ascending=False).reset_index().head(15))

print(top_15)

In [ ]:
# Комментарий ревьюера 2
tmp = df.copy()          # создаем копию текущего датасета
name_tmp = 'му-му'       # задаем имя столбца для проверки уникальных категорий
print(f'Заведения с одинаковым именем {name_tmp} относятся к разным категориям:\n\
{", ".join(tmp[tmp.name == name_tmp].category.unique())}')

In [ ]:
#график топ-15 популярных сетей в Москве
top_15.plot(kind='bar',
            x='name', y='name_count',
                title='Топ-15 популярных сетей в Москве',
                legend=False,
                ylabel='Рейтинг',
                xlabel='Заведения',
                rot=90)
plt.show()

In [ ]:
#Рейтинг заведений, которые входят в топ-15 сетей в Москве
top_15.plot(kind='barh',
            x='name', y='avg_rating',
                title='Рейтинг заведений, которые входят в топ-15 сетей в Москве',
                legend=False,
                ylabel='Рейтинг',
                xlabel='Заведения',
                rot=0)
plt.show()

- Самые популярные заведения в Москве - "Шоколадница" (категория кофейня) и "Домино'с_пицца" (категория пиццерия), "Додо пицца" (категория пиццерия).
- Значения рейтинга для Топ-15 заведений в Москве выше 3.9

---

### Задача 8

Изучите вариацию среднего чека заведения (столбец `middle_avg_bill`) в зависимости от района Москвы. Проанализируйте цены в Центральном административном округе и других. Как удалённость от центра влияет на цены в заведениях? Результат сопроводите подходящими визуализациями.


In [ ]:
df['middle_avg_bill'].hist(bins=50)

plt.title('Распределение значений среднего чека')
plt.xlabel('Средний чек')
plt.ylabel('Частота')

plt.show()

In [ ]:
df.boxplot(column='middle_avg_bill', vert=False)
plt.title('Распределение значений среднего чека')
plt.xlabel('Средний чек')

plt.show()

- Распределение значений среднего чека асимметричное, скошенное вправо.

In [ ]:
df['middle_avg_bill'].describe()

- Среднее значение (958) и медиана (750) находятся достаточно далеко друг от друга, что указывает на несимметричное распределение данных и наличие выраженных выбросов
- Минимальный ср чек - 0. Данные выбросы могут быть связаны с отсутствием внесенных данных, а также с различными акциями, бартерным сотрудничеством.
- Максимальный ср чек - 35000, что может быть связано с единичным большим чеком в дорогостоящем ресторане, либо с неправильно заполненной информацией о ср.чеке, что также является выбросами.
- Значения меньшя 25го процентиля (375) и выше 75го процентиля (1250) - можно считать выбросами.

In [ ]:
bill = df.groupby('district')['middle_avg_bill'].agg(['mean', 'median'])
print(bill)

In [ ]:
bill= df.groupby('district')['middle_avg_bill'].agg(['mean', 'median'])
bill.plot(kind='barh',
                title='Зависимость среднего чека от района',
                legend=True,
                ylabel='Средний чек заведения',
                xlabel='Район',
                rot=0)

plt.show()

- Самый высокий средний чек наблюдается в Центральном районе Москвы (1191.1)
- Далее чек снижается в зависимости от удаленности района:
  - В Западном, Северном и Северо-Западном районе также наблюдается высокий средний чек (1053.2, 927.9 и 822.2 соответственно)
  - Самый низкий средний чек наблюдается в Юго-Восточном (654.1), Северо-Востояном (716.6) и Юго-Западном (792.6) районах.

---


---

### Промежуточный вывод


- По Москве больше всего заведений представлены в категории кафе, на втором месте - ресторан, на третем месте - кофейня. Доля кафе составляет 28%, доля ресторанов - 24%, доля кофеен - 17%. Меньше всего заведений в категориях столовая и булочная, доля которых составляет 4% и 3% соответственно.
- Большая доля заведений находится в Центральном районе Москвы и составляет 27%. Самое маленькое кол-во заведений находится в Северно-Западном районе Москвы (доля заведений 5%)
- Самое маленькое кол-во заведений находится в Северно-Западном районе Москвы. 
- В центральном районе топ-3 категории заведений: ресторан, кафе, кофейня. 
- Самые низкие позиции по кол-ву заведений в центральном районе занимают: булочная, столовая, быстрое питание и пиццерия.

- В разрезе всех данных по количеству больше несетевых заведений, чем сетевых
- Категории заведений, которые чаще являются сетевыми: булочная (доля - 61%), пиццерия (доля - 52%), кофейня (доля - 51%)
  
- Средний рейтинг по Москве практически одинаков для всех категорий заведений и составлет 4+
- Наивысший рейтинг у заведений в категории бар/паб (4.39), ниже всего рейтинг в категории быстрое питание (4.05)

- Наибольшая корреляция рейтинга наблюдается с категорией цен в заведениях (0.22), административным районом (0.20) и категорией заведений (0.19). Для остальных признаков коэффициент корреляции снижается с 0.15 до 0.
- Коэффициент корреляции 0.22 означает слабо-положительную связь между ценовой категорией и рейтингом. 
- Между категорией заведения и категорией цены самая сильная связь (0.58), что означает, что тип заведения сильно влияет на его ценовую категорию. 

- Выделены Топ-15 популярных сетей в Москве. 
- Самые популярные заведения в Москве - "Шоколадница" (категория кофейня) и "Домино'с_пицца" (категория пиццерия), "Додо пицца" (категория пиццерия).
- Значения рейтинга для Топ-15 заведений в Москве выше 3.9

- Самый высокий средний чек наблюдается в Центральном районе Москвы (1191.1)
- Далее чек снижается в зависимости от удаленности района:
  - В Западном, Северном и Северо-Западном районе также наблюдается высокий средний чек (1053.2, 927.9 и 822.2 соответственно)
  - Самый низкий средний чек наблюдается в Юго-Восточном (654.1), Северо-Востояном (716.6) и Юго-Западном (792.6) районах.

## 4. Итоговый вывод и рекомендации



**На этапе предобработки данных:**

Были загружены и объеденены данные из 2х датасетов /datasets/rest_info.csv и /datasets/rest_price.csv. Объендененный датасет содержит 13 столбцов и 8406 строк, в которых представлена информация о заведениях общественного питания и о среднем чеке в данных заведениях.

- В шести стобцах (hours, seats, price, avg_bill, middle_avg_bill и middle_coffee_cup) были обнаружены пропущенные значения.
- Максимальное значение пропущенных данных в столбце 'middle_coffee_cup' - 93,6%, на втором месте столбец 'middle_avg_bill' - 62,5%, на третем месте столбец 'price' - 60,5%.
- Для оптимизации работы с данными в датафрейме были произведены следующие изменения типов данных:
  - 'chain': тип данных изменён с int64 на int8.
  - 'price': тип данных изменён с object на category.
- Проведена нормализация данных с текстовыми значениями с целью исключения неявных дубликатов, а также выявлено 4 явных дубликата.
- После удаления явных дубликатов осталось 8402 строк.
- Создан столбец is_24_7 с обозначением того, что заведение работает ежедневно и круглосуточно.

**При исследовательсом анализе выявлено:**

- По Москве больше всего заведений представлены в категории кафе, на втором месте - ресторан, на третем месте - кофейня. Доля кафе составляет 28%, доля ресторанов - 24%, доля кофеен - 17%. Меньше всего заведений в категориях столовая и булочная, доля которых составляет 4% и 3% соответственно.
- Большая доля заведений находится в Центральном районе Москвы и составляет 27%. Самое маленькое кол-во заведений находится в Северно-Западном районе Москвы (доля заведений 5%)
- Самое маленькое кол-во заведений находится в Северно-Западном районе Москвы. 
- В центральном районе топ-3 категории заведений: ресторан, кафе, кофейня. 
- Самые низкие позиции по кол-ву заведений в центральном районе занимают: булочная, столовая, быстрое питание и пиццерия.

- В разрезе всех данных по количеству больше несетевых заведений, чем сетевых
- Категории заведений, которые чаще являются сетевыми: булочная (доля - 61%), пиццерия (доля - 52%), кофейня (доля - 51%)
  
- Средний рейтинг по Москве практически одинаков для всех категорий заведений и составлет 4+
- Наивысший рейтинг у заведений в категории бар/паб (4.39), ниже всего рейтинг в категории быстрое питание (4.05)

- Наибольшая корреляция рейтинга наблюдается с категорией цен в заведениях (0.22), административным районом (0.20) и категорией заведений (0.19). Для остальных признаков коэффициент корреляции снижается с 0.15 до 0.
- Коэффициент корреляции 0.22 означает слабо-положительную связь между ценовой категорией и рейтингом. 
- Между категорией заведения и категорией цены самая сильная связь (0.58), что означает, что тип заведения сильно влияет на его ценовую категорию. 

- Выделены Топ-15 популярных сетей в Москве. 
- Самые популярные заведения в Москве - "Шоколадница" (категория кофейня) и "Домино'с_пицца" (категория пиццерия), "Додо пицца" (категория пиццерия).
- Значения рейтинга для Топ-15 заведений в Москве выше 3.9

- Самый высокий средний чек наблюдается в Центральном районе Москвы (1191.1)
- Далее чек снижается в зависимости от удаленности района:
  - В Западном, Северном и Северо-Западном районе также наблюдается высокий средний чек (1053.2, 927.9 и 822.2 соответственно)
  - Самый низкий средний чек наблюдается в Юго-Восточном (654.1), Северо-Востояном (716.6) и Юго-Западном (792.6) районах.

**Рекомендации на основе анализа данных:**

- Для открытия кафе можно рассмотреть Центральный административнй окуруг, так как там самый высокий средний чек, но также и самая высокая конкуренция по количеству открытых заведений, что может негативно сказаться на бизнесе. 
- Или можно рассмотреть менее загруженный по кол-ву заведений административный округ, также с относительно высоким средним чеком (Западный, Северный или Северо-Западный административный округ).
- Можно рассмотреть открытие заведения в категории кафе или рестораны, так как они самые востребованные. 
- Для бизнеса в Москве важно будет иметь показатель рейтинга заведения больше 4, поэтому нужно сделать акцент на качестве и маркентинге. 

- Второй путь открытия бизнеса: выбрать админимстративный округ с наименьшем количеством открытых заведений, не самую перегруженную конкуренцией категорию и занять нишу своим бизнесом с акцентом на качество, маркетинг и конкурентные цены для данного округа. 